# Step 1: Import Libraries

In [3]:
!pip install -q langchain langchain-community langchain-openai faiss-cpu pypdf sentence-transformers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 335.6/335.6 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [14]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
!pip uninstall -y langchain langchain-core langchain-community
!pip install langchain==0.2.14 langchain-community==0.2.12 langchain-core==0.2.36

Found existing installation: langchain 0.2.14
Uninstalling langchain-0.2.14:
  Successfully uninstalled langchain-0.2.14
Found existing installation: langchain-core 0.2.36
Uninstalling langchain-core-0.2.36:
  Successfully uninstalled langchain-core-0.2.36
Found existing installation: langchain-community 0.2.12
Uninstalling langchain-community-0.2.12:
  Successfully uninstalled langchain-community-0.2.12
  Using cached langchain-0.2.14-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_community-0.2.12-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_core-0.2.36-py3-none-any.whl.metadata (6.2 kB)
Using cached langchain-0.2.14-py3-none-any.whl (997 kB)
Using cached langchain_community-0.2.12-py3-none-any.whl (2.3 MB)
Using cached langchain_core-0.2.36-py3-none-any.whl (395 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt

# Step 2: Upload Document

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline

from google.colab import files

# Step 3: Upload Document

In [34]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print("Uploaded:", file_name)

Saving National Health Policy 2017 (English) .pdf to National Health Policy 2017 (English) .pdf
Uploaded: National Health Policy 2017 (English) .pdf


# Step 4: Load Document

In [35]:
loader = PyPDFLoader(file_name)
documents = loader.load()

print("Total pages:", len(documents))

Total pages: 42


# Step 5: Text Chunking

In [36]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)
print("Total chunks:", len(docs))

Total chunks: 249


# Step 6: Create Embeddings + Vector Store

In [37]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Step 7: Load LLM (FLAN-T5 BASE)

In [38]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA

# Install compatible version (run once, then restart runtime if needed)
!pip install -q transformers==4.36.2 langchain langchain-community

# Imports
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline

# Model name
model_name = "google/flan-t5-base"   # ✅ lightweight model

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Define a custom generation function to bypass transformers.pipeline task issues
def custom_generator(prompt, model, tokenizer, max_new_tokens, temperature):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        num_return_sequences=1,
        do_sample=True, # Allow sampling based on temperature
        top_k=50,       # For diverse outputs
        top_p=0.95      # For diverse outputs
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

# Create a minimal pipeline-like object that wraps the custom generator
class MinimalPipeline:
    def __init__(self, model, tokenizer, max_new_tokens, temperature):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.task = "text2text-generation" # Add the missing 'task' attribute

    def __call__(self, text_inputs, **kwargs):
        # HuggingFacePipeline expects a list of dictionaries with 'generated_text'
        if isinstance(text_inputs, list):
            results = []
            for prompt in text_inputs:
                generated_text = custom_generator(prompt, self.model, self.tokenizer, self.max_new_tokens, self.temperature)
                results.append([{'generated_text': generated_text}])
            return results
        else:
            generated_text = custom_generator(text_inputs, self.model, self.tokenizer, self.max_new_tokens, self.temperature)
            return [{'generated_text': generated_text}]

# Instantiate the minimal pipeline object
min_pipe = MinimalPipeline(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.5
)

# Convert to LangChain LLM
llm = HuggingFacePipeline(pipeline=min_pipe)

# Test run
response = llm.invoke("Explain in simple terms: What is artificial intelligence?")
print(response)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Artificial intelligence (AI) is the development of artificial intelligence.


# Step 8: Build RAG QA System

In [39]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

# Step 9: Ask Questions

In [40]:
def ask_question(query):
    result = qa_chain({"query": query})

    print("\n🟢 Answer:\n")
    print(result["result"])

    print("\n📚 Source Chunks:\n")
    for doc in result["source_documents"]:
        print("-", doc.page_content[:200], "...\n")

# Step 10: Run Example Queries

In [41]:
ask_question("Summarize this policy in simple language")
ask_question("What are the risks mentioned?")
ask_question("What are the key rules?")


🟢 Answer:

National Health Policy 2017

📚 Source Chunks:

- 36
National Health Policy 2017
need to be clearly spelt out.  The policy while supporting the need for moving in 
the direction of a rights based approach to healthcare is conscious of the fact that 
 ...

- at sub-divisional level in a cluster of few blocks. To achieve this, policy therefore 
aims: 
o To have at least two beds per thousand population distributed in such a 
way that it is accessible withi ...

- 28
National Health Policy 2017
safety, medical technologies, medical products, clinical trials, research and 
implementation of other health related laws - needs urgent and concrete steps 
towards ref ...


🟢 Answer:

Physical, chemical, and other workplace hazards

📚 Source Chunks:

- 8
National Health Policy 2017
Recognizing the risks arising from physical, chemical, and other workplace hazards, 
the policy advocates for providing greater focus on occupational health. Work-sites 
 ...

- regarding standards of care,